# Companies House MCP & Agent tools tests    

Companies House mcp ran locally and in Cloud Run

Purpose is to 
1. create a Agent tool that connects to Companies HouseMCP Server
2. Retreives and summarises relevant details requested (requires HITL to make the inital request)
3. HITL middleware/interrupts ask questions of the user where there is uncertainty (e.g. the name of the company not an exact match, or not found or address or director details non matching)


In [7]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient

from langgraph.graph import StateGraph
from langgraph.graph import START, END
from langgraph.types import interrupt, Command
from schemas.schemas import State, StateInput
from utils import parse_gmail, format_for_display, format_gmail_markdown

import os
import shutil
from dotenv import load_dotenv
import json
load_dotenv()


True

In [4]:
#!export PATH="$HOME/google-cloud-sdk/bin:$PATH"


## MCP - connect to Local service

In [8]:
COMPANIES_HOUSE_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY")

In [3]:
# Find npx path - subprocess may not have same PATH as notebook
npx_path = shutil.which("npx") or "/opt/homebrew/bin/npx"

# Ensure PATH includes homebrew bin for subprocess
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"

client = MultiServerMCPClient(
    {
        "companies-house": {
            "command": npx_path,  # Use full path to npx
            "transport": "stdio",
            "args": ["-y", "companies-house-mcp-server"],
            "env": {
                **env,  # Include PATH in environment
                "COMPANIES_HOUSE_API_KEY": COMPANIES_HOUSE_API_KEY
            }
        }
    }
)
tools = await client.get_tools()



In [ ]:
# 37 is a lot! Might want to filter down to just the essentials
tools_by_name = [tool.name for tool in tools]
tools_by_name


['search_companies',
 'get_company_profile',
 'get_registered_office_address',
 'get_registers',
 'get_insolvency',
 'get_exemptions',
 'get_uk_establishments',
 'advanced_company_search',
 'search_all',
 'search_officers',
 'search_disqualified_officers',
 'alphabetical_search',
 'dissolved_search',
 'get_officers',
 'get_officer_appointment',
 'get_corporate_officer_disqualification',
 'get_natural_officer_disqualification',
 'get_officer_appointments_list',
 'get_filing_history',
 'get_filing_history_item',
 'get_charges',
 'get_charge_details',
 'get_persons_with_significant_control',
 'get_psc_corporate_entity_beneficial_owner',
 'get_psc_corporate_entity',
 'get_psc_individual_beneficial_owner',
 'get_psc_individual',
 'get_psc_individual_verification',
 'get_psc_individual_full_record',
 'get_psc_legal_person_beneficial_owner',
 'get_psc_legal_person',
 'get_psc_statement',
 'get_psc_statements_list',
 'get_psc_super_secure_beneficial_owner',
 'get_psc_super_secure',
 'get_docum

In [4]:
agent = create_agent(
    "gpt-4o",
    tools  
)
ch_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "get directors for Goodman's Consulting"}]}
)

In [ ]:
ch_response

{'messages': [HumanMessage(content="get directors for Goodman's Consulting", additional_kwargs={}, response_metadata={}, id='4536422c-297f-41ab-ab60-116da4f55ff3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 1962, 'total_tokens': 1979, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CtgSEtnD2wBgNqk3t6aSfjbMsz9mz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b807d-7e8d-75a0-af69-fb1e6ee85e1f-0', tool_calls=[{'name': 'search_companies', 'args': {'query': "Goodman's Consulting"}, 'id': 'call_hhwkxCPHfTacMtQzXMzpVS0H', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1962,

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## MCP Service running on CloudRun 

In [6]:
#! pip install google-auth google-auth-oauthlib google-auth-httplib2 requests

In [9]:
import importlib
import gcp.python_client_iam_mcp
importlib.reload(gcp.python_client_iam_mcp) 

<module 'gcp.python_client_iam_mcp' from '/Users/stevegoodman/dev/fionaa-be/src/gcp/python_client_iam_mcp.py'>

In [10]:
CH_MCP_SERVICE_URL = os.environ["CH_MCP_SERVICE_URL"]

# Set GOOGLE_APPLICATION_CREDENTIALS to the correct path
from pathlib import Path
credentials_path = Path.cwd().parent / ".secrets" / "fionaa-service-acct.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())

client = gcp.python_client_iam_mcp.IAMAuthenticatedMCPClient(CH_MCP_SERVICE_URL)

try:
    # Health check
    print("Checking service health...")
    health = client.health_check()
    print(f"Health: {json.dumps(health, indent=2)}\n")
    
    # List available tools
    print("Fetching available tools...")
    tools_response = client.list_tools()
    tools = tools_response.get("result", {}).get("tools", [])
    print(f"Found {len(tools)} tools\n")
    
    # Show first few tools
    print("Sample tools:")
    for tool in tools[:5]:
        print(f"  - {tool['name']}: {tool['description'][:60]}...")
    
    # Example: Search for companies
    print("\nSearching for companies...")
    search_result = client.call_tool("search_companies", {
        "query": "Goodmans Consulting",
        "items_per_page": 5
    })
    
    if "error" in search_result:
        print(f"Error: {search_result['error']}")
    else:
        result_content = search_result.get("result", {}).get("content", [])
        if result_content:
            print("Search results:")
            print(json.dumps(json.loads(result_content[0]["text"]), indent=2))
    
except Exception as e:
    # Print the full error message (which may contain helpful instructions)
    error_msg = str(e)
    if "401 Unauthorized" in error_msg or "roles/run.invoker" in error_msg:
        # This is a permission error with helpful instructions
        print(error_msg)
    else:
        # Generic error
        print(f"Error: {error_msg}")
    sys.exit(1)

Checking service health...
Health: {
  "status": "ok",
  "service": "companies-house-mcp"
}

Fetching available tools...
Found 37 tools

Sample tools:
  - search_companies: Search for companies by name or company number...
  - get_company_profile: Get detailed profile information for a specific company...
  - get_registered_office_address: Get the registered office address of a company...
  - get_registers: Get company registers information...
  - get_insolvency: Get company insolvency information...

Searching for companies...
Search results:
{
  "items": [
    {
      "kind": "searchresults#company",
      "description_identifier": [
        "dissolved-on"
      ],
      "company_status": "dissolved",
      "date_of_creation": "2012-07-11",
      "date_of_cessation": "2017-12-28",
      "company_type": "ltd",
      "company_number": "08139267",
      "address": {
        "address_line_1": "Heskin",
        "locality": "Preston",
        "postal_code": "PR7 5PA",
        "premises": "

# Test Calling the tool from an LLM

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="you are a helpful assistant that can answer questions about the companies house"
)
agent.invoke({"messages": [HumanMessage(content="Tell me about the directors of company Goodmans Consulting")]})

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content='Tell me about the directors of company Goodmans Consulting', additional_kwargs={}, response_metadata={}, id='f4af4945-3f02-43c6-bcc1-14785d625600'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 468, 'prompt_tokens': 807, 'total_tokens': 1275, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cwq54O8sdZAtubt3oLg6NKci28PNQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bad4f-f837-7cb3-9080-38332c57f25b-0', tool_calls=[{'name': 'search_companies', 'args': {}, 'id': 'call_vdb2YTp03rbITjLi6mz6Fsst', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
# This is an interesting response - How would we turn this into an interrupt so the humanan can clarify?


##  Langhain and/or langgraph workflow

HITL langchain middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [119]:
config = {"configurable": {"thread_id": "123"}}

agent.invoke({'messages':HumanMessage("read an email for id 1")}, config=config)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content='read an email for id 1', additional_kwargs={}, response_metadata={}, id='3024b45f-e549-49ba-b23f-ba56e6bddbde'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 86, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CtfmNzhgsP2DCuuwf9X8ZhtZwmMDU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b8055-ef65-7801-bf85-1750909e7919-0', tool_calls=[{'name': 'read_email_tool', 'args': {'email_id': '1'}, 'id': 'call_fccfkuUjAF340TAhGvxSko5M', 'type': 'tool_call'}], usage_metadata={'input_tokens': 86, 'output_tokens': 16, 'total_tokens': 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [122]:
agent.invoke({'messages':HumanMessage("sent an email to steve@hotmail with subject line 'hi'")}, config=config)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content="sent an email to steve@hotmail with subject line 'hi'", additional_kwargs={}, response_metadata={}, id='83c5760f-035b-4f1b-8a0c-4c8ce4f0c773'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 92, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CtfsD0DuFtLCy0zo15sA78v5VsJ0Z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b805b-7557-7603-9873-6e7c11f4e099-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'steve@hotmail', 'subject': 'hi', 'body': ''}, 'id': 'call_7qcduCSQaKh71EpMjAAAb30e', 'type': 'tool_call'}], 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [123]:
response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [124]:
response

{'messages': [HumanMessage(content="sent an email to steve@hotmail with subject line 'hi'", additional_kwargs={}, response_metadata={}, id='83c5760f-035b-4f1b-8a0c-4c8ce4f0c773'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 92, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CtfsD0DuFtLCy0zo15sA78v5VsJ0Z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b805b-7557-7603-9873-6e7c11f4e099-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'steve@hotmail', 'subject': 'hi', 'body': ''}, 'id': 'call_7qcduCSQaKh71EpMjAAAb30e', 'type': 'tool_call'}], 

In [ ]:
from langchain_core.tools import tool



def email_response(state: State) -> str:
    """
    Given an email, return a response to the email.
    """
    prompt = f"""
    You are a helpful assistant that can help with email responses. Please respond to the email.
    The email is as follows:
    {state["email_input"]["body"]}
    """
    
    messages = [llm.invoke(prompt)]
    # Create interrupt that is shown to the user
    request = {
        "action_request": {
            "action": f"Email Assistant: {state['email_input']['body']}",
            "args": {}
        },
        "config": {
            "allow_ignore": True,  
            "allow_respond": True, 
            "allow_edit": False, 
            "allow_accept": True,  
        },
        # Email to show in Agent Inbox
        "description": messages.content
    }

    # Agent Inbox responds with a list of dicts with a single key `type` that can be `accept`, `edit`, `ignore`, or `response`.  
    response = interrupt([request])[0]

    # If user provides feedback, go to response agent and use feedback to respond to email   
    if response["type"] == "response":
        # Add feedback to messages 
        # Used by the response agent
        messages.append({"role": "user",
                        "content": f"User wants to reply to the email. Use this feedback to respond: user_input"
                        })
        # Go to response agent
        
        goto = "response_agent"

    # If user ignores email, go to END
    elif response["type"] == "accept":
        messages.append({"role": "user",
                        "content": f"User wants to reply to the email. with {draft_reply}"
                        })
        print("ACCEPTED REGISTERED")
        goto = END
    elif response["type"] == "ignore":
        goto = END
    update = {
     "messages": messages, "goto": goto
    }

    return Command(resume=[{"type": "response", "args": {"update": [draft_reply]}}])

In [99]:
from langchain_core.messages import HumanMessage
def process_email_input(state: State) -> str:
    ''' logic is to extract the email body and the email subject
    and then pass these to the agent.
    '''

    author, to, subject, body, email_id = parse_gmail(state["email_input"])
    return {
        "messages": [HumanMessage(content=f"Processing email: {state['email_input']['body']}")]
    }
    
    
s1 = State(StateInput(email_input={"to": "steve@goodman.com", "from": "fiona@goodman.com", "subject": "test", "body": "test", "id": "test"}))   

process_email_input(s1)



!Email_input from Gmail!
{'to': 'steve@goodman.com', 'from': 'fiona@goodman.com', 'subject': 'test', 'body': 'test', 'id': 'test'}


{'messages': [HumanMessage(content='Processing email: test', additional_kwargs={}, response_metadata={})]}

In [100]:
graph = (StateGraph(State, input=StateInput)
    .add_node("process_email_input", process_email_input)
    .add_node("email_response", email_response)
    
    .add_edge(START, "process_email_input")
    .add_edge("process_email_input", "email_response")
    .add_edge("email_response", END)
)



/var/folders/4b/dthvd31s2jnbxsmqfg30w0w00000gp/T/ipykernel_10668/1508824416.py:1: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  graph = (StateGraph(State, input=StateInput)


In [101]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
email={"to": "steve@goodman.com", "from": "fiona@goodman.com", "subject": "test", "body": "test", "id": "test"}
checkpointer = InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)
thread_id_2 = uuid.uuid4()
thread_config_2 = {"configurable": {"thread_id": thread_id_2}}



#workflow.invoke({"email_input": email}, config=thread_config_2)

for chunk in workflow.stream({"email_input": email}, config=thread_config_2):
    # Inspect interrupt object if present
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

!Email_input from Gmail!
{'to': 'steve@goodman.com', 'from': 'fiona@goodman.com', 'subject': 'test', 'body': 'test', 'id': 'test'}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AttributeError: 'list' object has no attribute 'content'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [97]:
from langgraph.types import Command

print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']}")
for chunk in workflow.stream(Command(resume=[{"type": "accept"}]), config=thread_config_2):
    # Inspect interrupt object if present
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")


Simulating user accepting the {'action': 'Email Assistant: test', 'args': {}}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AttributeError: 'AIMessage' object has no attribute 'append'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [92]:
chunk['email_response']['messages'][-1]

AIMessage(content='Here are a few quick reply options you can use depending on the tone you want. The message content is just “test,” so these are generic confirmations you can adapt.\n\nShort and casual\n- Thanks for the test message — I’ve got it. Let me know if you need anything else.\n\nNeutral and polite\n- Hello, I’ve received your test email. Please let me know if there’s anything I should review or if you’re ready to share the real content.\n\nNeed more details\n- Thanks for the test. Could you share the topic or purpose so I can tailor my response?\n\nFormal\n- Dear [Name], I have received your test message. Please confirm receipt and advise how you would like me to proceed. Best regards, [Your Name]\n\nSubject line ideas (if you’re replying via email)\n- Re: Test message received\n- Acknowledgement of test email\n- Confirmation of test message\n\nIf you share a bit more context (who the sender is, your relationship, and what you want to accomplish), I can tailor a single, pol